In [17]:
using PlotlyJS
using LaTeXStrings
include("./processing.jl")
include("../model/post_processing.jl")
include("../model/utils/empirical_mu.jl")
dim = (1000,500)
g_BLUE = "1616A7"
g_GREY = "#7F7F7F"
g_SAVE = false
THRES = 0.001
FOLDER_PATH = joinpath("..","output");

In [ ]:
function add_fields(df, kwargs...)
    for (k, v) in kwargs
        df[!, Symbol(k)] .= v
    end
    cols = vcat([Symbol(k) for (k, _) in kwargs], names(df)[.!([Symbol(k) for (k, _) in kwargs] .∈ names(df))])
    return df[:, cols]
end


function load_solutions(solution_name::String, solution_folder::String, days::Vector{Int}; kwargs...)
    """
    Load and combine solution data from multiple parquet files.

    This function reads solution data from parquet files located in the specified 
    folder, combines them into a single dictionary of DataFrames, and applies 
    additional keyword arguments to each DataFrame.

    Parameters:
    -----------
    solution_name : String
        The name of the solution to load.
    solution_folder : String
        The folder where the solution parquet files are located.
    days : Vector{Int}
        A vector of day identifiers to load. Each day identifier will be prefixed 
        with 'n_' to form the filename.
    kwargs : NamedTuple
        Additional keyword arguments to add as columns to each DataFrame in the 
        resulting dictionary.

    Returns:
    --------
    Dict{String, DataFrame}
        A dictionary where keys are the combined keys from all solutions and 
        values are the concatenated DataFrames for each key.

    Example:
    --------
    >>> load_solutions("solution1", "/path/to/solutions", [1, 2], extra_col="value")
    Dict("key1" => DataFrame1, "key2" => DataFrame2, ...)
    """
    solution_keys = get(kwargs, :solution_keys, nothing)
    solutions = [parquet_to_solution(solution_name, joinpath(solution_folder, "n_$(d)"), solution_keys) for d in days]
    return combine_solutions(solutions; kwargs...)
end

function combine_solutions(solutions; kwargs...)
    solutions = filter(!isempty, solutions)  # cleaning empty
    all_keys = union([keys(s) for s in solutions]...)
    solutions_dict = Dict(k => vcat([s[k] for s in solutions if haskey(s, k)]...) for k in all_keys) #TODO: fix vcat that is not merging DFs with different columns
    for (k, v) in kwargs
        for df in values(solutions_dict)
            df[!, Symbol(k)] .= v
        end
    end
    return solutions_dict
end

# ρs = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.99]
ρs = [0.1, 0.2, 0.3]
s_57_4 = [Dict("solution_folder" => "solutions_v57.$ρ.4s", "ρ" => ρ, "VLGEN" => 30, "model_type" => "envelope") for ρ in ρs]
s_58_4 = [Dict("solution_folder" => "solutions_v58.$ρ.4s", "ρ" => ρ, "VLGEN" => 1e-6, "model_type" => "e-reserve") for ρ in ρs]
s_s57_7_7 = [Dict("solution_folder" => "solutions_v_s57.$ρ.7.7s", "ρ" => ρ, "VLGEN" => 1000, "model_type" => "stochastic") for ρ in ρs]

s_uc = []
s_ed = []
ss = vcat(s_57_4, s_58_4, s_s57_7_7)
# ss = s_s57_7_7
for sol in ss
    ρ = sol["ρ"]
    s = sol["solution_folder"]
    # s_uc_name = sol["model_type"] == "stochastic" ? "s_suc" : "s_uc"

    if sol["model_type"] == "stochastic"
        s_uc_name = "s_suc"
    else
        s_uc_name = "s_uc"
        s_ed_ = load_solutions("s_ed", joinpath("..", "output", s), [7], model_type = sol["model_type"], VLGEN=sol["VLGEN"], ρ=ρ, solution_id = s)
        push!(s_ed, s_ed_)
    end 
    s_uc_ = load_solutions(s_uc_name, joinpath("..", "output", s), [7], model_type = sol["model_type"], VLGEN=sol["VLGEN"], ρ=ρ, solution_id = s)
    push!(s_uc, s_uc_)
    
end
s_uc = combine_solutions(s_uc) 
s_ed = combine_solutions(s_ed)

reading...
../output/solutions_v57.0.1.4s/n_7/s_ed_demand.parquet
../output/solutions_v57.0.1.4s/n_7/s_ed_generation.parquet
../output/solutions_v57.0.1.4s/n_7/s_ed_storage.parquet
../output/solutions_v57.0.1.4s/n_7/s_ed_reserve.parquet
../output/solutions_v57.0.1.4s/n_7/s_ed_scalar.parquet
../output/solutions_v57.0.1.4s/n_7/s_ed_generation_parameters.parquet
../output/solutions_v57.0.1.4s/n_7/s_ed_storage_parameters.parquet
../output/solutions_v57.0.1.4s/n_7/s_ed_objective_function.parquet
../output/solutions_v57.0.1.4s/n_7/s_ed_dual_variables.parquet
...done


LoadError: DimensionMismatch: arrays could not be broadcast to a common size: a has axes Base.OneTo(9) and b has axes Base.OneTo(18)

6-element Vector{Any}:
 Dict{Symbol, DataFrame}(:generation => 134400×16 DataFrame
    Row │ r_id   hour    production_MW  curtailment_MW  commit     start       ⋯
        │ Int64  Int64?  Float64?       Float64?        Float64?   Float64?    ⋯
────────┼───────────────────────────────────────────────────────────────────────
      1 │     3     145        607.375             0.0        1.0        0.0   ⋯
      2 │     4     145         30.019             0.0        1.0        0.0
      3 │     5     145         49.686             0.0        1.0        0.0
      4 │     6     145        571.645             0.0        1.0        0.0
      5 │     7     145          0.0               0.0        1.0        0.0   ⋯
      6 │     8     145          0.0               0.0        1.0        0.0
      7 │     9     145          0.0               0.0        0.0        0.0
      8 │    10     145          0.0               0.0        0.0        0.0
      9 │    11     145          0.0              

In [ ]:
day = 7
input_folder = "../input/base_case_increased_storage_energy_v8.0.8.4"
gen_df, loads_multi_df, random_loads_multi_df, gen_variable_multi_df, storage_df, required_reserve = load_deterministic_data(day, input_folder)
required_energy_reserve = load_energy_reserve(day, input_folder, loads_multi_df, gen_variable_multi_df)
;

In [ ]:

temportal_weights = false
model, sol_1, sol_2 = solve_empirical_µ_get_solution(gen_df, loads_multi_df, storage_df, required_reserve, required_energy_reserve; temportal_weights = temportal_weights)
;


In [ ]:
function calculate_mu_t(sol_1_)
    sol_1 = dropmissing(sol_1_)
    mu = DataFrame()
    for (θ, RES) in [(:θUPDIS, :RESUPDIS), (:θUPCH, :RESUPCH), (:θDNDIS, :RESDNDIS), (:θDNCH, :RESDNCH)]
        mu[!, Symbol("$θ/$RES")] = sol_1[:, θ] ./ sol_1[:, RES]
    end
    mu[!, :t] = sol_1.t
    mu = mu[1:end, :]
    return unstack(stack(mu, Not(:t)),:t, :value)
end
mu_t = calculate_mu_t(sol_1)

In [ ]:
legend_attr = attr(
    x=0.01,
    y=0,
    yanchor="bottom",
    xanchor="left",
    orientation="V"
    )
layout_args = Dict(attr(
    plot_bgcolor="rgba(0,0,0,0)",
    # font = attr(size=17),
    yaxis=attr(showgrid=false, gridwidth=0.11, gridcolor = "grey", showline=true, linecolor="grey", mirror = true),
    xaxis=attr(showgrid=false, gridwidth=0.11, gridcolor = "grey", showline=true, linecolor="grey", mirror = true),
    width=dim[1]/2, height=dim[2],
    yaxis_title="Power [MW]", xaxis_title="Hour",
    showlegend=true,
    ))

ks = [:θUPDIS,:RESUPDIS, :θDNCH,:RESDNCH]
p = plot(stack(sol_1, ks), y = :value, group=:r_id, color =:variable, facet_col = :r_id, Layout(layout_args, legend = legend_attr), mode="lines+markers")


In [ ]:
if g_SAVE savefig(p, "reserve.pdf", width=Int(dim[1]/2), height= dim[2]) end

In [ ]:
legend_attr = attr(
        x=1,
        y=0,
        yanchor="bottom",
        xanchor="right",
        orientation="v"
    )
layout_args = Dict(attr(
    plot_bgcolor="rgba(0,0,0,0)",
    # font = attr(size=17),
    yaxis=attr(showgrid=false, gridwidth=0.11, gridcolor = "grey", showline=true, linecolor="grey", mirror = true),
    xaxis=attr(showgrid=false, gridwidth=0.11, gridcolor = "grey", showline=true, linecolor="grey", mirror = true),
    width=dim[1]/2, height=dim[2],
    yaxis_title="Energy [MWh]", xaxis_title="Hour",
    showlegend=true,
    ))
    
cumsum_sol_1 = combine(groupby(sol_1, :r_id), [:RESUPDIS, :RESDNDIS, :RESUPCH, :RESDNCH, :θUPDIS, :θUPCH, :θDNCH, :θDNDIS] .=> cumsum, renamecols = false)
cumsum_sol_1.t = unique(sol_1.t)
leftjoin!(cumsum_sol_1, sol_1[:, [:r_id, :t, :MAXERESUPDIS, :MAXERESUPCH, :MAXERESDNCH, :MAXERESDNDIS]], on = [:r_id, :t])


ks = [:θUPDIS, :RESUPDIS, :θDNCH, :RESDNCH, :MAXERESUPDIS,:MAXERESDNCH]
line_styles = Dict(:θUPDIS => "dot", :RESUPDIS => "dash", :MAXERESUPDIS => "solid")
p = plot(stack(cumsum_sol_1, ks), y = :value, group=:r_id, color =:variable, Layout(layout_args, legend = legend_attr), line=attr(widthdash = get(line_styles, :variable, "solid")) , mode="lines+markers")


In [ ]:
if g_SAVE savefig(p, "cumulative_reserve.pdf", width=Int(dim[1]/2), height= dim[2]) end

In [ ]:
# mu_t_all = CSV.read("mu_t_all.csv", DataFrame)
aux = CSV.read("mu_t_all.csv", DataFrame)
stacked_mu_t_all = stack(aux, Not([:rho, :mu]), variable_name=:t, value_name=:value)
mu_t_all = unstack(stacked_mu_t_all, :mu, :value)

In [ ]:
legend_attr = attr(
        x=0,
        y=0,
        yanchor="bottom",
        xanchor="left",
        orientation="v"
    )
layout_args = Dict(attr(
    plot_bgcolor="rgba(0,0,0,0)",
    # font = attr(size=17),
    yaxis=attr(showgrid=false, gridwidth=0.11, gridcolor = "grey", showline=true, linecolor="grey", mirror = true),
    xaxis=attr(showgrid=false, gridwidth=0.11, gridcolor = "grey", showline=true, linecolor="grey", mirror = true),
    width=dim[1]/2, height=dim[2],
    yaxis_title="θUPDIS/RESUPDIS", xaxis_title="Hour",
    showlegend=true,
    ))
    
p = plot(mu_t_all, y = Symbol("θUPDIS/RESUPDIS"), color = :rho, Layout(layout_args, legend = legend_attr), mode="lines+markers")

In [ ]:
if g_SAVE savefig(p, "mu_up_dis.pdf", width=Int(dim[1]/2), height= dim[2]) end

In [ ]:
legend_attr = attr(
        x=0,
        y=0,
        yanchor="bottom",
        xanchor="left",
        orientation="v"
    )
layout_args = Dict(attr(
    plot_bgcolor="rgba(0,0,0,0)",
    # font = attr(size=17),
    yaxis=attr(showgrid=false, gridwidth=0.11, gridcolor = "grey", showline=true, linecolor="grey", mirror = true),
    xaxis=attr(showgrid=false, gridwidth=0.11, gridcolor = "grey", showline=true, linecolor="grey", mirror = true),
    width=dim[1]/2, height=dim[2],
    yaxis_title="θDNCH/RESNCH", xaxis_title="Hour",
    showlegend=true,
    ))
    
p = plot(mu_t_all, y = Symbol("θDNCH/RESDNCH"), color = :rho, Layout(layout_args, legend = legend_attr), mode="lines+markers")

In [ ]:
if g_SAVE savefig(p, "mu_dn_ch.pdf", width=Int(dim[1]/2), height= dim[2]) end